In [1]:
import numpy as np
import sys
from sklearn.ensemble import GradientBoostingClassifier
import pandas as pd
import os
from tqdm import tqdm
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from dacs_core.vanillaBH import bh
from dacs_core.diverseSelect import cluster_exact_diverseSelect
import time

In [2]:
def one_hot_encode_column(df, column_name):
    """
    Replaces a column in a DataFrame with its one-hot encoded columns.

    Parameters:
    - df: pd.DataFrame, the input DataFrame.
    - column_name: str, the column to one-hot encode.

    Returns:
    - pd.DataFrame, the DataFrame with the specified column replaced by one-hot encoded columns.
    """
    # Ensure the column exists
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame.")
    
    # One-hot encode the specified column
    one_hot_encoded = pd.get_dummies(df[column_name], prefix=column_name).astype(int)
    
    # Drop the original column and concatenate the one-hot encoded columns
    df = df.drop(columns=[column_name])
    df = pd.concat([df, one_hot_encoded], axis=1)
    
    return df


In [3]:
df = pd.read_csv('Placement_Data_Full_Class.csv')
df = df[['gender', 'ssc_p', 'ssc_b', 'hsc_p', 'hsc_b', \
    'hsc_s', 'degree_p', 'degree_t', 'workex', 'etest_p',\
    'specialisation', 'mba_p', 'status']]

df = df.dropna(axis=0)

In [4]:
gender_map = {'M': 0, 'F': 1}
ssc_b_map = {'Others': 0, 'Central': 1}
hsc_b_map = {'Others': 0, 'Central': 1}
hsc_s_map = {'Commerce': 0, 'Science': 1, 'Arts': 2}
degree_t_map = {'Sci&Tech': 0, 'Comm&Mgmt': 1, 'Others': 2}
workex_map = {'No': 0, 'Yes': 1}
specialisation_map = {'Mkt&HR': 0, 'Mkt&Fin': 1}
status_map = {'Placed': 1, 'Not Placed': 0}

# cluster at the factor level

df['gender'] = df['gender'].map(gender_map)
df['ssc_b'] = df['ssc_b'].map(ssc_b_map)
df['hsc_b'] = df['hsc_b'].map(hsc_b_map)
df['hsc_s'] = df['hsc_s'].map(hsc_s_map)
df['degree_t'] = df['degree_t'].map(degree_t_map)
df['workex'] = df['workex'].map(workex_map)
df['specialisation'] = df['specialisation'].map(specialisation_map)
df['status'] = df['status'].map(status_map)

df = pd.get_dummies(df, columns=['hsc_s', 'degree_t'], drop_first=True).astype(int)

In [5]:
from sklearn.calibration import CalibratedClassifierCV

diversification_variable = 'gender'

X = df.drop(['status', 'gender'], axis=1).to_numpy()
Z = df[diversification_variable].to_numpy()
y = df['status'].to_numpy()

# cutoff = np.quantile(y, 0.5)
# y -= cutoff

# SETTING 1: More calib than test
dataset_length = len(y)
print(dataset_length)
ntrain = 45
n = 125
m = 45

print(len(X), len(y))

ncomponents = df[diversification_variable].nunique()

nsims = 1
skip=1
np.random.seed(42)
def get_scores(X, Y, mu_hat):
    return np.where(Y > 0, np.inf, -mu_hat.predict_proba(X)[:,1]+0.00000001*np.random.normal(size=X.shape[0]))


setting1_freqs = []
setting1_vanilla_freqs = []
setting1_diversities = []
setting1_vanilla_diversities = []
setting1_fdrs = []
setting1_vanilla_fdrs = []
setting1_powers = []
setting1_vanilla_powers = []
setting1_overall_num_rejections = []
setting1_vanilla_overall_num_rejections = []
setting1_p0s = []
setting1_vanilla_p0s = []

setting1_serr_freqs = []
setting1_serr_vanilla_freqs = []
setting1_serr_diversities = []
setting1_serr_vanilla_diversities = []
setting1_serr_fdrs = []
setting1_serr_vanilla_fdrs = []
setting1_serr_powers = []
setting1_serr_vanilla_powers = []
setting1_serr_overall_num_rejections = []
setting1_serr_vanilla_overall_num_rejections = []

# CHANGE LATER
alpha_list = [0.25]#np.linspace(0.05, 0.35, 50)
for alpha in alpha_list:
    fdps = []
    vanilla_fdps = []

    tdps = []
    vanilla_tdps = []

    total_num_rejections = []
    vanilla_total_num_rejections = []

    total_counts = np.zeros((nsims,ncomponents))
    vanilla_total_counts = np.zeros((nsims,ncomponents))
    for trial_iterator in tqdm(range(nsims)):
        perm = np.random.permutation(dataset_length)

        trainX = X[perm][:ntrain,:]
        calibX = X[perm][ntrain:ntrain+n,:]
        testX = X[perm][ntrain+n:,:]

        calibZ = Z[perm][ntrain:ntrain+n]
        testZ = Z[perm][ntrain+n:]
        
        trainY = y[perm][:ntrain]
        calibY = y[perm][ntrain:ntrain+n]
        testY = y[perm][ntrain+n:]

        muHat = GradientBoostingClassifier(learning_rate=0.1, max_depth=3, \
                                           n_estimators=50, subsample=0.7, random_state = 0)
        muHat.fit(np.array(trainX), np.array(trainY))

        # fit classifier, too
        base = GradientBoostingClassifier(
            learning_rate=0.1, max_depth=3, n_estimators=50, subsample=0.7, random_state=0
        )
        clf = CalibratedClassifierCV(base, method='sigmoid', cv=3)
        # clf = make_pipeline(
        #     StandardScaler(),
        #     LogisticRegression(C=0.1, max_iter=5000)
        # )

        clf.fit(trainX, trainY)

        combinedX = np.concatenate((calibX,testX))
        log_likelihoods = np.log(clf.predict_proba(combinedX)[:, 0])
        # log_likelihoods=np.log(0.5)*np.ones(n+m)
        
        # add noise because using gradient boosting on trees
        calibS = get_scores(calibX, calibY, muHat)
        testS = get_scores(testX, np.zeros(m), muHat)

        with open(os.devnull, 'w') as devnull:
            original_stdout = sys.stdout
            original_stderr = sys.stderr
            sys.stdout = devnull
            sys.stderr = devnull
            try:
                rejections = cluster_exact_diverseSelect(calibS, testS, n, m, alpha, np.concatenate((calibZ,testZ)), \
                                            ncomponents, skip, log_likelihoods=log_likelihoods)

                vanilla_rejections, _, __ = bh(calibS, testS, n, m, alpha)
            finally:
                sys.stdout = original_stdout
                sys.stderr = original_stderr
        

        counts = np.bincount(testZ[rejections.astype(bool)], \
                            minlength=ncomponents)
        vanilla_counts = np.bincount(testZ[vanilla_rejections.astype(bool)],\
                                minlength=ncomponents)
        
        fdp = np.sum([int(testY[i] <= 0)*int(rejections[i] == 1.) \
                                for i in range(m)])/max(1., np.sum(rejections))
        tdp = np.sum([int(testY[i] > 0)*int(rejections[i] == 1.) \
                        for i in range(m)])/max(1., np.sum([int(testY[i] > 0) \
                                                        for i in range(m)]))
        num_rejections = np.sum(rejections)

        metrics = [fdp, tdp, num_rejections]


        vanilla_fdp = np.sum([int(testY[i] <= 0)*int(vanilla_rejections[i] == 1.) \
                                for i in range(m)])/max(1., np.sum(vanilla_rejections))
        vanilla_tdp = np.sum([int(testY[i] > 0)*int(vanilla_rejections[i] == 1.) \
                        for i in range(m)])/max(1., np.sum([int(testY[i] > 0) \
                                                        for i in range(m)]))
        vanilla_num_rejections = np.sum(vanilla_rejections)

        vanilla_metrics = [vanilla_fdp, vanilla_tdp, vanilla_num_rejections]

        total_counts[trial_iterator] = np.copy(counts)
        vanilla_total_counts[trial_iterator] = np.copy(vanilla_counts)

        fdps.append(fdp)
        vanilla_fdps.append(vanilla_fdp)
        tdps.append(tdp)
        vanilla_tdps.append(vanilla_tdp)
        total_num_rejections.append(num_rejections)
        vanilla_total_num_rejections.append(vanilla_num_rejections)


    freq_arr = []
    vanilla_freq_arr = []

    p0 = 0.
    vanilla_p0 = 0.

    diversity_arr = []
    vanilla_diversity_arr = []


    for trial_iterator in range(nsims):
        if total_num_rejections[trial_iterator] > 0:
            freq_arr.append(total_counts[trial_iterator]/total_counts[trial_iterator].sum())
            diversity_arr.append(np.min(total_counts[trial_iterator])/np.sum(total_counts[trial_iterator]))
        else:
            p0 += 1.
            diversity_arr.append(-0.5)
        if vanilla_total_num_rejections[trial_iterator] > 0:
            vanilla_freq_arr.append(vanilla_total_counts[trial_iterator]/vanilla_total_counts[trial_iterator].sum())
            vanilla_diversity_arr.append(np.min(vanilla_total_counts[trial_iterator])/np.sum(vanilla_total_counts[trial_iterator]))
        else:
            vanilla_p0 += 1.
            vanilla_diversity_arr.append(-0.5)

    diversity_arr = np.array(diversity_arr)
    vanilla_diversity_arr = np.array(vanilla_diversity_arr)
    freq_arr = np.array(freq_arr)
    vanilla_freq_arr = np.array(vanilla_freq_arr)

    print(f"Results for alpha = {alpha}")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print(f'Proportions: {np.mean(freq_arr, axis=0)}')
    print(f'Vanilla proportions: {np.mean(vanilla_freq_arr, axis=0)}')
    print(f'Diversity metric: {np.mean(diversity_arr)}')
    print(f'Vanilla diversity metric: {np.mean(vanilla_diversity_arr)}')
    print(f'Proportion of zero rejections: {p0/nsims}')
    print(f'Vanilla roportion of zero rejections: {vanilla_p0/nsims}')
    print(f'FDR: {np.mean(fdps)}')
    print(f'Vanilla FDR: {np.mean(vanilla_fdps)}')
    print(f'Power: {np.mean(tdps)}')
    print(f'Vanilla power: {np.mean(vanilla_tdps)}')
    print(f'Avg num rejections: {np.mean(total_num_rejections)}')
    print(f'Vanilla avg num rejections: {np.mean(vanilla_total_num_rejections)}')
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print()
    print()

    setting1_freqs.append(np.mean(freq_arr, axis=0))
    setting1_serr_freqs.append(np.std(freq_arr, axis=0)/np.sqrt(len(freq_arr)))
    setting1_vanilla_freqs.append(np.mean(vanilla_freq_arr, axis=0))
    setting1_serr_vanilla_freqs.append(np.std(vanilla_freq_arr, axis=0)/np.sqrt(len(vanilla_freq_arr)))
    setting1_diversities.append(np.mean(diversity_arr))
    setting1_serr_diversities.append(np.std(diversity_arr)/np.sqrt(len(diversity_arr)))
    setting1_vanilla_diversities.append(np.mean(vanilla_diversity_arr))
    setting1_serr_vanilla_diversities.append(np.std(vanilla_diversity_arr)/np.sqrt(len(vanilla_diversity_arr)))
    setting1_p0s.append(p0/nsims)
    setting1_vanilla_p0s.append(vanilla_p0/nsims)
    setting1_fdrs.append(np.mean(fdps))
    setting1_serr_fdrs.append(np.std(fdps)/np.sqrt(len(fdps)))
    setting1_vanilla_fdrs.append(np.mean(vanilla_fdps))
    setting1_serr_vanilla_fdrs.append(np.std(vanilla_fdps)/np.sqrt(len(vanilla_fdps)))
    setting1_powers.append(np.mean(tdps))
    setting1_serr_powers.append(np.std(tdps)/np.sqrt(len(tdps)))
    setting1_vanilla_powers.append(np.mean(vanilla_tdps))
    setting1_serr_vanilla_powers.append(np.std(vanilla_tdps)/np.sqrt(len(vanilla_tdps)))
    setting1_overall_num_rejections.append(np.mean(total_num_rejections))
    setting1_serr_overall_num_rejections.append(np.std(total_num_rejections)/np.sqrt(len(total_num_rejections)))
    setting1_vanilla_overall_num_rejections.append(np.mean(vanilla_total_num_rejections))
    setting1_serr_vanilla_overall_num_rejections.append(np.std(vanilla_total_num_rejections)/np.sqrt(len(vanilla_total_num_rejections)))



215
215 215


100%|██████████| 1/1 [00:00<00:00,  3.97it/s]

Results for alpha = 0.25
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Proportions: [0.55 0.45]
Vanilla proportions: [0.73809524 0.26190476]
Diversity metric: 0.45
Vanilla diversity metric: 0.2619047619047619
Proportion of zero rejections: 0.0
Vanilla roportion of zero rejections: 0.0
FDR: 0.05
Vanilla FDR: 0.2619047619047619
Power: 0.59375
Vanilla power: 0.96875
Avg num rejections: 20.0
Vanilla avg num rejections: 42.0
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~




In [6]:
from sklearn.calibration import CalibratedClassifierCV

diversification_variable = 'gender'

X = df.drop(['status', 'gender'], axis=1).to_numpy()
Z = df[diversification_variable].to_numpy()
y = df['status'].to_numpy()

# cutoff = np.quantile(y, 0.5)
# y -= cutoff

# SETTING 1: More calib than test
dataset_length = len(y)
print(dataset_length)
ntrain = 45
n = 125
m = 45

print(len(X), len(y))

ncomponents = df[diversification_variable].nunique()

nsims = 1000
skip=1
np.random.seed(42)
def get_scores(X, Y, mu_hat):
    return np.where(Y > 0, np.inf, -mu_hat.predict_proba(X)[:,1]+0.00000001*np.random.normal(size=X.shape[0]))


setting1_freqs = []
setting1_vanilla_freqs = []
setting1_diversities = []
setting1_vanilla_diversities = []
setting1_fdrs = []
setting1_vanilla_fdrs = []
setting1_powers = []
setting1_vanilla_powers = []
setting1_overall_num_rejections = []
setting1_vanilla_overall_num_rejections = []
setting1_p0s = []
setting1_vanilla_p0s = []

setting1_serr_freqs = []
setting1_serr_vanilla_freqs = []
setting1_serr_diversities = []
setting1_serr_vanilla_diversities = []
setting1_serr_fdrs = []
setting1_serr_vanilla_fdrs = []
setting1_serr_powers = []
setting1_serr_vanilla_powers = []
setting1_serr_overall_num_rejections = []
setting1_serr_vanilla_overall_num_rejections = []

# CHANGE LATER
alpha_list = [0.25]#np.linspace(0.05, 0.35, 50)
for alpha in alpha_list:
    fdps = []
    vanilla_fdps = []

    tdps = []
    vanilla_tdps = []

    total_num_rejections = []
    vanilla_total_num_rejections = []

    total_counts = np.zeros((nsims,ncomponents))
    vanilla_total_counts = np.zeros((nsims,ncomponents))
    for trial_iterator in tqdm(range(nsims)):
        perm = np.random.permutation(dataset_length)

        trainX = X[perm][:ntrain,:]
        calibX = X[perm][ntrain:ntrain+n,:]
        testX = X[perm][ntrain+n:,:]

        calibZ = Z[perm][ntrain:ntrain+n]
        testZ = Z[perm][ntrain+n:]
        
        trainY = y[perm][:ntrain]
        calibY = y[perm][ntrain:ntrain+n]
        testY = y[perm][ntrain+n:]

        muHat = GradientBoostingClassifier(learning_rate=0.1, max_depth=3, \
                                           n_estimators=50, subsample=0.7, random_state = 0)
        muHat.fit(np.array(trainX), np.array(trainY))

        # fit classifier, too
        base = GradientBoostingClassifier(
            learning_rate=0.1, max_depth=3, n_estimators=50, subsample=0.7, random_state=0
        )
        clf = CalibratedClassifierCV(base, method='sigmoid', cv=3)
        # clf = make_pipeline(
        #     StandardScaler(),
        #     LogisticRegression(C=0.1, max_iter=5000)
        # )

        clf.fit(trainX, trainY)

        combinedX = np.concatenate((calibX,testX))
        # log_likelihoods = np.log(clf.predict_proba(combinedX)[:, 0])
        log_likelihoods=np.log(0.5)*np.ones(n+m)
        
        # add noise because using gradient boosting on trees
        calibS = get_scores(calibX, calibY, muHat)
        testS = get_scores(testX, np.zeros(m), muHat)

        with open(os.devnull, 'w') as devnull:
            original_stdout = sys.stdout
            original_stderr = sys.stderr
            sys.stdout = devnull
            sys.stderr = devnull
            try:
                rejections = cluster_exact_diverseSelect(calibS, testS, n, m, alpha, np.concatenate((calibZ,testZ)), \
                                            ncomponents, skip, log_likelihoods=None)

                vanilla_rejections, _, __ = bh(calibS, testS, n, m, alpha)
            finally:
                sys.stdout = original_stdout
                sys.stderr = original_stderr
        

        counts = np.bincount(testZ[rejections.astype(bool)], \
                            minlength=ncomponents)
        vanilla_counts = np.bincount(testZ[vanilla_rejections.astype(bool)],\
                                minlength=ncomponents)
        
        fdp = np.sum([int(testY[i] <= 0)*int(rejections[i] == 1.) \
                                for i in range(m)])/max(1., np.sum(rejections))
        tdp = np.sum([int(testY[i] > 0)*int(rejections[i] == 1.) \
                        for i in range(m)])/max(1., np.sum([int(testY[i] > 0) \
                                                        for i in range(m)]))
        num_rejections = np.sum(rejections)

        metrics = [fdp, tdp, num_rejections]


        vanilla_fdp = np.sum([int(testY[i] <= 0)*int(vanilla_rejections[i] == 1.) \
                                for i in range(m)])/max(1., np.sum(vanilla_rejections))
        vanilla_tdp = np.sum([int(testY[i] > 0)*int(vanilla_rejections[i] == 1.) \
                        for i in range(m)])/max(1., np.sum([int(testY[i] > 0) \
                                                        for i in range(m)]))
        vanilla_num_rejections = np.sum(vanilla_rejections)

        vanilla_metrics = [vanilla_fdp, vanilla_tdp, vanilla_num_rejections]

        total_counts[trial_iterator] = np.copy(counts)
        vanilla_total_counts[trial_iterator] = np.copy(vanilla_counts)

        fdps.append(fdp)
        vanilla_fdps.append(vanilla_fdp)
        tdps.append(tdp)
        vanilla_tdps.append(vanilla_tdp)
        total_num_rejections.append(num_rejections)
        vanilla_total_num_rejections.append(vanilla_num_rejections)


    freq_arr = []
    vanilla_freq_arr = []

    p0 = 0.
    vanilla_p0 = 0.

    diversity_arr = []
    vanilla_diversity_arr = []


    for trial_iterator in range(nsims):
        if total_num_rejections[trial_iterator] > 0:
            freq_arr.append(total_counts[trial_iterator]/total_counts[trial_iterator].sum())
            diversity_arr.append(np.min(total_counts[trial_iterator])/np.sum(total_counts[trial_iterator]))
        else:
            p0 += 1.
            diversity_arr.append(-0.5)
        if vanilla_total_num_rejections[trial_iterator] > 0:
            vanilla_freq_arr.append(vanilla_total_counts[trial_iterator]/vanilla_total_counts[trial_iterator].sum())
            vanilla_diversity_arr.append(np.min(vanilla_total_counts[trial_iterator])/np.sum(vanilla_total_counts[trial_iterator]))
        else:
            vanilla_p0 += 1.
            vanilla_diversity_arr.append(-0.5)

    diversity_arr = np.array(diversity_arr)
    vanilla_diversity_arr = np.array(vanilla_diversity_arr)
    freq_arr = np.array(freq_arr)
    vanilla_freq_arr = np.array(vanilla_freq_arr)

    print(f"Results for alpha = {alpha}")
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print(f'Proportions: {np.mean(freq_arr, axis=0)}')
    print(f'Vanilla proportions: {np.mean(vanilla_freq_arr, axis=0)}')
    print(f'Diversity metric: {np.mean(diversity_arr)}')
    print(f'Vanilla diversity metric: {np.mean(vanilla_diversity_arr)}')
    print(f'Proportion of zero rejections: {p0/nsims}')
    print(f'Vanilla roportion of zero rejections: {vanilla_p0/nsims}')
    print(f'FDR: {np.mean(fdps)}')
    print(f'Vanilla FDR: {np.mean(vanilla_fdps)}')
    print(f'Power: {np.mean(tdps)}')
    print(f'Vanilla power: {np.mean(vanilla_tdps)}')
    print(f'Avg num rejections: {np.mean(total_num_rejections)}')
    print(f'Vanilla avg num rejections: {np.mean(vanilla_total_num_rejections)}')
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")
    print()
    print()

    setting1_freqs.append(np.mean(freq_arr, axis=0))
    setting1_serr_freqs.append(np.std(freq_arr, axis=0)/np.sqrt(len(freq_arr)))
    setting1_vanilla_freqs.append(np.mean(vanilla_freq_arr, axis=0))
    setting1_serr_vanilla_freqs.append(np.std(vanilla_freq_arr, axis=0)/np.sqrt(len(vanilla_freq_arr)))
    setting1_diversities.append(np.mean(diversity_arr))
    setting1_serr_diversities.append(np.std(diversity_arr)/np.sqrt(len(diversity_arr)))
    setting1_vanilla_diversities.append(np.mean(vanilla_diversity_arr))
    setting1_serr_vanilla_diversities.append(np.std(vanilla_diversity_arr)/np.sqrt(len(vanilla_diversity_arr)))
    setting1_p0s.append(p0/nsims)
    setting1_vanilla_p0s.append(vanilla_p0/nsims)
    setting1_fdrs.append(np.mean(fdps))
    setting1_serr_fdrs.append(np.std(fdps)/np.sqrt(len(fdps)))
    setting1_vanilla_fdrs.append(np.mean(vanilla_fdps))
    setting1_serr_vanilla_fdrs.append(np.std(vanilla_fdps)/np.sqrt(len(vanilla_fdps)))
    setting1_powers.append(np.mean(tdps))
    setting1_serr_powers.append(np.std(tdps)/np.sqrt(len(tdps)))
    setting1_vanilla_powers.append(np.mean(vanilla_tdps))
    setting1_serr_vanilla_powers.append(np.std(vanilla_tdps)/np.sqrt(len(vanilla_tdps)))
    setting1_overall_num_rejections.append(np.mean(total_num_rejections))
    setting1_serr_overall_num_rejections.append(np.std(total_num_rejections)/np.sqrt(len(total_num_rejections)))
    setting1_vanilla_overall_num_rejections.append(np.mean(vanilla_total_num_rejections))
    setting1_serr_vanilla_overall_num_rejections.append(np.std(vanilla_total_num_rejections)/np.sqrt(len(vanilla_total_num_rejections)))



215
215 215


  5%|▍         | 49/1000 [00:05<01:46,  8.89it/s]


KeyboardInterrupt: 